In [2]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

    BiocManager::install(c("ensembldb", "AnnotationHub"))

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'BiocVersion', 'ensembldb', 'AnnotationHub'

also installing the dependencies ‘matrixStats’, ‘abind’, ‘SparseArray’, ‘formatR’, ‘MatrixGenerics’, ‘S4Arrays’, ‘DelayedArray’, ‘lambda.r’, ‘futile.options’, ‘png’, ‘SummarizedExperiment’, ‘cigarillo’, ‘RCurl’, ‘rjson’, ‘futile.logger’, ‘snow’, ‘BH’, ‘XVector’, ‘lazyeval’, ‘UCSC.utils’, ‘KEGGREST’, ‘XML’, ‘GenomicAlignments’, ‘BiocIO’, ‘restfulr’, ‘bitops’, ‘BiocParallel’, ‘Rhtslib’, ‘filelock’, ‘BiocGenerics’, ‘GenomicRanges’, ‘GenomicFeatures’, ‘AnnotationFilter’, ‘RSQLite’, ‘Biobase’, ‘Seqinfo’, ‘GenomeInfoDb’, ‘AnnotationDbi’, ‘rtracklayer’, ‘S4Vectors’, ‘Rsamtools’, ‘IRange

In [3]:
library(ensembldb)
library(AnnotationHub)

ah <- AnnotationHub()
edb <- ah[["AH119325"]]

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘pac

downloading 1 resources

retrieving 1 resource



loading from cache



In [4]:
# Get all transcripts of your gene
tx <- transcripts(edb, filter = ~ gene_id == "ENSG00000128573")

# Extract only the IDs
tx_ids <- tx$tx_id

# Save to file
write.table(tx_ids, "transcripts.tsv", sep="\t", row.names=FALSE, col.names=FALSE, quote=FALSE)

# Preview
print(tx_ids)

 [1] "ENST00000412402" "ENST00000440349" "ENST00000703612" "ENST00000703613"
 [5] "ENST00000703614" "ENST00000703615" "ENST00000441290" "ENST00000703616"
 [9] "ENST00000635638" "ENST00000495516" "ENST00000634411" "ENST00000635109"
[13] "ENST00000634623" "ENST00000393494" "ENST00000350908" "ENST00000462331"
[17] "ENST00000408937" "ENST00000393498" "ENST00000393495" "ENST00000403559"
[21] "ENST00000459666" "ENST00000378237" "ENST00000393489" "ENST00000635534"
[25] "ENST00000360232" "ENST00000452963" "ENST00000390668" "ENST00000703617"
[29] "ENST00000635563" "ENST00000634372" "ENST00000703618" "ENST00000634664"


In [5]:
# Get all proteins
pr <- proteins(edb, filter = ~ gene_id == "ENSG00000128573")

# Extract only the IDs
pr_ids <- pr$protein_id

# Save to file
write.table(pr_ids, "proteins.tsv", sep="\t", row.names=FALSE, col.names=FALSE, quote=FALSE)

# Preview
print(pr_ids)

 [1] "ENSP00000405470" "ENSP00000395552" "ENSP00000515396" "ENSP00000515397"
 [5] "ENSP00000515398" "ENSP00000515399" "ENSP00000416825" "ENSP00000515400"
 [9] "ENSP00000489073" "ENSP00000489135" "ENSP00000489457" "ENSP00000488944"
[13] "ENSP00000377132" "ENSP00000265436" "ENSP00000418100" "ENSP00000386200"
[17] "ENSP00000377135" "ENSP00000377133" "ENSP00000385069" "ENSP00000367482"
[21] "ENSP00000377129" "ENSP00000489229" "ENSP00000353367" "ENSP00000409826"
[25] "ENSP00000375084" "ENSP00000515401" "ENSP00000488954" "ENSP00000515402"


In [6]:
# Check protein lengths this way instead
pr_df <- as.data.frame(pr)
print(head(pr_df))
print(colnames(pr_df))

            tx_id      protein_id
1 ENST00000412402 ENSP00000405470
2 ENST00000440349 ENSP00000395552
3 ENST00000703612 ENSP00000515396
4 ENST00000703613 ENSP00000515397
5 ENST00000703614 ENSP00000515398
6 ENST00000703615 ENSP00000515399
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              protein_sequence
1                          

In [7]:
# Build IRanges correctly - one range per protein
pr_iranges <- IRanges(
    start = rep(1, nrow(pr_df)),
        end   = pr_df$protein_length,   # we'll fix this column name after you check
            names = pr_df$protein_id
            )

            # Run mapping
            mapping <- proteinToGenome(pr_iranges, edb)

            # Save
            mapping_df <- do.call(rbind, lapply(mapping, as.data.frame))
            write.table(mapping_df, "protein_genome_map.tsv",
                        sep="\t", row.names=FALSE, quote=FALSE)
                        cat("Done!")

Fetching CDS for 28 proteins ... 
28 found

Checking CDS and protein sequence lengths ... 
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000515399'. The returned genomic coordinates might thus not be correct for this protein."
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000488944'. The returned genomic coordinates might thus not be correct for this protein."
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000377133'. The returned genomic coordinates might thus not be correct for this protein."
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000409826'. The returned genomic coordinates might thus not be correct for this protein."
Warning message:
"Could not find a CDS whith the expected length for protein: 'ENSP00000488954'. The returned genomic coordinates might thus not be correct for this protein."
Warning message:
"Could not find a

Done!